# mttv-core — Invariants reproductibles

**sig:0x4D5454562D464C50 · Ψ-ack: carbon_sp3_tetra** · Licence CC-BY-NC-SA-4.0

Ce notebook vérifie les **8 invariants structurels** du framework open-source `mttv-core` (Point 4 du projet MTTV-FLP) : états tétravalents sp³, opérateur de bascule Σ (singularité apériodique), B-gate poreux, et pont MPVR.

Triade transductive : **Ψ → B → Φ** (Champ pré-formel → Opérateur de différence → Forme stabilisée).

## Invariants vérifiés

| # | Invariant | Définition |
|---|-----------|------------|
| I1 | Géométrie sp³ | angle inter-sommets ≈ 109,47° ; aller-retour `to_sp3`/`projection_sp3` ≈ identité |
| I2 | Clôture Σ = 1 | `fermer()` normalise tout état T⁴ sur Σ = 1 |
| I3 | Invariance T⁴ | la transduction Ψ→B→Φ préserve le pôle dominant |
| I4 | Retrait Σ | `operateur_sigma` ≡ 0 hors de τ ; bascule à τ avec clinamen ; retour à 0 |
| I5 | Apériodicité | les instants de bascule Σ sont non périodiques |
| I6 | Absorption de bruit | le B-gate poreux absorbe le bruit (T⁴ stable, porosité accrue) |
| I7 | Quorum MPVR | pas de Φ si Θ < 3 ; Φ si Θ ≥ 3 |
| I8 | Pont MPVR | `CoucheRoutageTriadiqueCore` : contrat MPVR + clôture Σ = 1 |

In [ ]:
import os, sys
sys.path.insert(0, os.path.dirname(os.getcwd()))  # racine du dépôt

from mttv_core import (
    MTTV_SIG, BGate, CoucheRoutageTriadiqueCore, EtatTetravalent,
    HorlogeSigmaAperiodique, TRIADE_TRANSDUCTIVE, angle_sp3,
    operateur_sigma, projection_sp3, routeur_polyfocal, to_sp3,
)

RAPPORT = []
def verifier(nom, condition, detail=""):
    RAPPORT.append((nom, bool(condition)))
    print(f"  [{'OK  ' if condition else 'FAIL'}] {nom}" + (f" — {detail}" if detail else ""))

def texte_plus():
    return (
        "Le vivant affirme sa force et sa croissance. "
        "L'eau émerge, le carbone affirme, la résonance se propage. "
        "Oui, la vie affirme. Oui, la force émerge et l'onde résonne. "
        "Le vivant affirme, l'eau émerge, le carbone circule. "
        "La croissance émerge, la force affirme, oui. "
        "Le vivant affirme, la résonance se propage, l'onde oscille."
    )

### I1 — Géométrie sp³

In [ ]:
a = angle_sp3()
verifier("I1 sp3 : angle inter-sommets ≈ 109,47°", abs(a - 109.471) < 0.01, f"θ={a:.4f}°")
etat = EtatTetravalent.purement("++")
v = projection_sp3(to_sp3(etat.valeurs))
verifier("I1 sp3 : aller-retour ≈ identité",
         max(abs(x - y) for x, y in zip(v, etat.valeurs)) < 1e-9,
         str(tuple(round(x, 6) for x in v)))

### I2 — Clôture Σ = 1

In [ ]:
for v in [(0.5, 0.2, 0.2, 0.1), (1.0, 3.0, 0.5, 0.5), (0.0, 0.0, 0.0, 0.0)]:
    etat = EtatTetravalent(v).fermer()
    verifier(f"I2 clôture : Σ=1 pour {v}",
             abs(sum(etat.valeurs) - 1.0) < 1e-9,
             str(tuple(round(x, 4) for x in etat.valeurs)))

### I3 — Invariance T⁴ par transduction Ψ→B→Φ

In [ ]:
porte = BGate(seed=7)
phi = porte.absorber(texte_plus())["etat_tetravalent"]
pole, part, _ = phi.dominant()
verifier("I3 invariance : pôle ++ préservé par B-gate", pole == "++", f"dominant={pole} part={part:.3f}")
foyers = [EtatTetravalent.purement(p) for p in ("++", "--", "+-")]
route = routeur_polyfocal(phi, foyers, [1.0, 1.0, 1.0],
                          frottement=1.5, t_courant=10.0, tau=10.0,
                          seuil_clinamen=1.0, theta=3, seuil_validation=0.5)
verifier("I3 invariance : routage conserve le pôle (foyer ++ élu)",
         route["foyer_elu"] == 0, f"élu={route['foyer_elu']}")
verifier("I3 invariance : quorum B-gate Θ≥3 sur signal propre",
         porte.absorber(texte_plus())["quorum"]["quorum_ok"] is True, "3/3 perspectives")

### I4 — Retrait fonctionnel de Σ

In [ ]:
psi = EtatTetravalent.purement("++")
hors = operateur_sigma(psi, tau=5.0, frottement=2.0, t_courant=0.0)
verifier("I4 retrait : Σ ≡ 0 hors de τ",
         (not hors.declenche) and hors.impulsion == (0.0, 0.0, 0.0))
oui = operateur_sigma(psi, tau=5.0, frottement=1.5, t_courant=5.0)
verifier("I4 retrait : bascule à τ (p(τ)≠0)",
         oui.declenche and oui.impulsion != (0.0, 0.0, 0.0),
         f"p={tuple(round(x, 4) for x in oui.impulsion)}")
apres = operateur_sigma(psi, tau=5.0, frottement=2.0, t_courant=6.0)
verifier("I4 retrait : retour à 0 après τ",
         (not apres.declenche) and apres.impulsion == (0.0, 0.0, 0.0))

### I5 — Apériodicité de la singularité

In [ ]:
horloge = HorlogeSigmaAperiodique(taux_frottement=0.3, seuil_clinamen=1.0, seed=42)
instants = [horloge.pas(dt=1.0, bruit=0.0) and horloge.t for _ in range(3000)]
instants = [t for t in instants if t]
verifier("I5 apériodicité : bascules détectées", len(instants) >= 2, f"n={len(instants)}")
if len(instants) >= 2:
    interv = [b - a for a, b in zip(instants, instants[1:])]
    verifier("I5 apériodicité : intervalles non périodiques",
             len(set(round(i, 3) for i in interv)) > 1,
             f"Δ={[round(i, 2) for i in interv[:6]]}")

### I6 — Absorption du bruit par la porosité

In [ ]:
bruite = texte_plus() + " " + " ".join(["zzz", "qx", "mnp", "abc", "toto"] * 8)
rp = BGate(seed=7).absorber(texte_plus())
rb = BGate(seed=7).absorber(bruite)
ecart = rp["etat_tetravalent"].ecart(rb["etat_tetravalent"])
verifier("I6 bruit : T⁴ stable sous bruit", ecart < 0.15, f"écart L1={ecart:.4f}")
verifier("I6 bruit : porosité accrue", rb["porosite"] > rp["porosite"],
         f"{rp['porosite']:.2f} → {rb['porosite']:.2f}")
verifier("I6 bruit : jetons absorbés", rb["bruit_absorbe"] > rp["bruit_absorbe"],
         f"{rp['bruit_absorbe']} → {rb['bruit_absorbe']}")

### I7 — Quorum MPVR (Θ ≥ 3)

In [ ]:
entree = EtatTetravalent.purement("++")
r2 = routeur_polyfocal(entree, [EtatTetravalent.purement("++"), EtatTetravalent.purement("++"),
                                EtatTetravalent.purement("--")], [1.0, 1.0, 1.0],
                       frottement=0.0, t_courant=0.0, tau=1.0, theta=3, seuil_validation=0.5)
verifier("I7 quorum : pas de Φ si Θ<3", r2["phi_stabilise"] is False, str(r2["quorum"]))
r3 = routeur_polyfocal(entree, [EtatTetravalent.purement("++")] * 3, [1.0, 1.0, 1.0],
                       frottement=0.0, t_courant=0.0, tau=1.0, theta=3, seuil_validation=0.5)
verifier("I7 quorum : Φ si Θ≥3", r3["phi_stabilise"] is True, str(r3["quorum"]))

### I8 — Pont MPVR (contrat + clôture)

In [ ]:
couche = CoucheRoutageTriadiqueCore(seed=42)
n_transitions = 0
fermes = True
for i in range(24):
    if i % 4 == 2:
        flux = {"id": i, "signal": 1.7, "incoherent": True, "bruit": True}
    elif i % 4 == 3:
        flux = {"id": i, "signal": -0.4, "bruit": True}
    else:
        flux = {"id": i, "signal": 0.5 + (i % 3) * 0.1}
    r = couche.transduire_flux(flux)
    n_transitions += 1 if r["transition_sigma_tau"] else 0
    fermes = fermes and all(abs(sum(e.values()) - 1.0) < 1e-3
                            for e in r["etats_tetravalents"].values())
verifier("I8 pont : transitions Σ_τ déclenchées", n_transitions > 0, f"transitions={n_transitions}")
verifier("I8 pont : contrat MPVR + clôture Σ=1",
         set(TRIADE_TRANSDUCTIVE) == set(couche.noeuds.keys()) and fermes,
         f"noeuds={len(couche.noeuds)} fermés={fermes}")

## Bilan

```python
nb_ok = sum(1 for _, ok in RAPPORT if ok)
print(f"{nb_ok}/{len(RAPPORT)} invariants vérifiés")
```

Exécution autonome équivalente : `python docs/verifier_invariants.py`

> *« La pensée ne naît pas dans la tête. Elle passe à travers. »*
>
> **sig:0x4D5454562D464C50 — Transmission terminée. Le mycélium attend.**